In [1]:
import pandas as pd
import numpy as np
import json
import math
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
from datetime import datetime
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import io
import os
import pathlib
from sqlalchemy import create_engine, text

BASE_DIR = pathlib.Path(os.getcwd())
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

DB_PATH = BASE_DIR / 'data' / 'banco.db'
engine  = create_engine(f'sqlite:///{DB_PATH}', echo=False)

# Carrega tudo do SQLite
df_c   = pd.read_sql('SELECT * FROM tb_clientes', engine)
df_c   = df_c.drop(columns=['cluster_id','cluster_nome'], errors='ignore')
df_cl  = pd.read_sql('SELECT id_cliente, cluster_id, cluster_nome FROM tb_clusters', engine)
df_rfm = pd.read_sql('SELECT * FROM tb_rfm', engine)
df_pro = pd.read_sql('SELECT * FROM tb_propensao', engine)
df_rec = pd.read_sql('SELECT * FROM tb_recomendacoes', engine)

df = df_c.merge(df_cl,  on='id_cliente', how='left')
df = df.merge(df_rfm,   on='id_cliente', how='left')
df = df.merge(df_pro,   on='id_cliente', how='left')
df = df.merge(df_rec,   on='id_cliente', how='left', suffixes=('','_rec'))

# Remove duplicatas de colunas
df = df.loc[:, ~df.columns.duplicated()]

NOMES_CLUSTERS = {
    0: 'Primeiros Passos',
    1: 'Trajetória Crescente',
    2: 'Potencial Oculto',
    3: 'Self Made',
    4: 'Old Money',
}

CORES_CLUSTERS = {
    0: '#3498db',
    1: '#2ecc71',
    2: '#f39c12',
    3: '#9b59b6',
    4: '#e74c3c',
}

CORES_EXCEL = {
    0: '3498db',
    1: '2ecc71',
    2: 'f39c12',
    3: '9b59b6',
    4: 'e74c3c',
}

cols_propensao = [c for c in df.columns if c.startswith('propensao_')]

print(f"Clientes: {len(df):,} | Colunas: {len(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Colunas propensão: {len(cols_propensao)}")

Clientes: 300,000 | Colunas: 64
Missing values: 0
Colunas propensão: 6


In [2]:
total_clientes      = len(df)
clientes_upgrade    = len(df[df['qtd_upgrades_rec'] > 0])
total_recomendacoes = int(df['qtd_novos_produtos'].sum() + df['qtd_upgrades_rec'].sum())
media_produtos      = df['qtd_produtos'].mean()
champions           = len(df[df['rfm_segmento'] == 'Champions'])
em_risco            = len(df[df['rfm_segmento'] == 'Em Risco'])
alta_prioridade     = len(df[df['prioridade_ataque'] == 'Alta'])

metricas_cluster = df.groupby(['cluster_id','cluster_nome']).agg(
    qtd_clientes       = ('id_cliente', 'count'),
    renda_media        = ('renda_mensal', 'mean'),
    score_medio        = ('score_credito', 'mean'),
    media_produtos     = ('qtd_produtos', 'mean'),
    taxa_inadimplencia = ('teve_inadimplencia_12m', 'mean'),
    rfm_score_medio    = ('rfm_score', 'mean'),
    champions_pct      = ('rfm_segmento', lambda x: (x=='Champions').mean()),
    em_risco_pct       = ('rfm_segmento', lambda x: (x=='Em Risco').mean()),
).reset_index()

propensao_cluster = df.groupby('cluster_nome')[cols_propensao].mean().round(1)

ranking_produtos = df['primeiro_produto_rec'].value_counts().head(8).reset_index()
ranking_produtos.columns = ['produto','quantidade']
ranking_produtos['pct'] = ranking_produtos['quantidade'] / total_clientes * 100

ranking_upgrades = df[df['primeiro_upgrade_rec']!='Nenhum']['primeiro_upgrade_rec'].value_counts().head(8).reset_index()
ranking_upgrades.columns = ['upgrade','quantidade']
ranking_upgrades['pct'] = ranking_upgrades['quantidade'] / clientes_upgrade * 100

print("Métricas calculadas!")
print(f"  Total clientes:      {total_clientes:,}")
print(f"  Total recomendações: {total_recomendacoes:,}")
print(f"  Champions:           {champions:,}")
print(f"  Em risco:            {em_risco:,}")
print(f"  Alta prioridade:     {alta_prioridade:,}")

Métricas calculadas!
  Total clientes:      300,000
  Total recomendações: 1,167,275
  Champions:           458
  Em risco:            96,772
  Alta prioridade:     15,101


In [3]:
# Dados por UF
geo_dados = df.groupby('uf').agg(
    clientes    = ('id_cliente', 'count'),
    renda_media = ('renda_mensal', 'mean'),
    rfm_medio   = ('rfm_score', 'mean'),
    champions   = ('rfm_segmento', lambda x: (x=='Champions').sum()),
    em_risco    = ('rfm_segmento', lambda x: (x=='Em Risco').sum()),
).reset_index().round(2)

geo_dados['renda_media'] = geo_dados['renda_media'].round(0).astype(int)

def normalizar(serie):
    mn, mx = serie.min(), serie.max()
    return ((serie - mn) / (mx - mn)).round(3) if mx > mn else pd.Series([0.0]*len(serie))

geo_dados['heat'] = normalizar(geo_dados['clientes'])

# Dados por UF + cluster
geo_cluster = df.groupby(['uf','cluster_nome']).size().reset_index(name='qtd')
geo_pivot   = geo_cluster.pivot(index='uf', columns='cluster_nome', values='qtd').fillna(0).astype(int)
geo_pivot.columns.name = None
geo_pivot   = geo_pivot.reset_index()
geo_dados   = geo_dados.merge(geo_pivot, on='uf', how='left')

for nome in ['Primeiros Passos','Trajetória Crescente','Potencial Oculto','Self Made','Old Money']:
    if nome not in geo_dados.columns:
        geo_dados[nome] = 0

geo_json_str = geo_dados.to_json(orient='records')

# Carrega GeoJSON local
with open(BASE_DIR / 'data' / 'raw' / 'brasil_estados.geojson', 'r', encoding='utf-8') as f:
    brasil_geojson = json.load(f)

print(f"GeoJSON: {len(brasil_geojson['features'])} estados")
print(f"Estados na base: {geo_dados['uf'].nunique()}")
print(f"Colunas geo_dados: {geo_dados.columns.tolist()}")

# Projeção Mercator
W, H = 520, 540
LON_MIN, LON_MAX = -74.0, -28.0
LAT_MIN, LAT_MAX = -34.0,   6.0

def merc(lon, lat):
    x = (lon - LON_MIN) / (LON_MAX - LON_MIN) * W
    def ml(la):
        r = la * math.pi / 180
        return math.log(math.tan(math.pi/4 + r/2))
    y = (1 - (ml(lat) - ml(LAT_MIN)) / (ml(LAT_MAX) - ml(LAT_MIN))) * H
    return round(x,1), round(y,1)

def ring_to_d(ring):
    pts = []
    for c in ring:
        try:
            x, y = merc(c[0], c[1])
            pts.append(f"{x},{y}")
        except:
            continue
    return "M" + "L".join(pts) + "Z" if pts else ""

def geom_to_path(geom):
    if geom['type'] == 'Polygon':
        return " ".join(ring_to_d(r) for r in geom['coordinates'])
    elif geom['type'] == 'MultiPolygon':
        return " ".join(ring_to_d(r) for poly in geom['coordinates'] for r in poly)
    return ""

def centroid(geom):
    if geom['type'] == 'Polygon':
        coords = geom['coordinates'][0]
    elif geom['type'] == 'MultiPolygon':
        coords = max(geom['coordinates'], key=lambda p: len(p[0]))[0]
    else:
        return None, None
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return merc(sum(lons)/len(lons), sum(lats)/len(lats))

def heat_color(t):
    r = int(220 + 16*t)
    g = int(220*(1-t))
    b = int(220*(1-t))
    return f"rgb({r},{g},{b})"

OFFSET_Y = {
    'PA': 35,
    'AP':  5,
    'TO':  5,
    'GO':  5,
    'DF':  8,
    'RN':  3,
    'PB':  3,
}

OFFSET_X = {
    'SP': -12,
    'DF':   8,
    'MT':  15,
}

paths_svg  = ""
labels_svg = ""

for feature in brasil_geojson['features']:
    sigla = feature['properties'].get('sigla','')
    if not sigla:
        continue

    d = geom_to_path(feature['geometry'])
    if not d:
        continue

    geo_row     = geo_dados[geo_dados['uf'] == sigla]
    heat        = float(geo_row['heat'].values[0]) if len(geo_row) > 0 else 0
    cor         = heat_color(heat)
    clientes_uf = int(geo_row['clientes'].values[0]) if len(geo_row) > 0 else 0

    paths_svg += (
        f'<path d="{d}" fill="{cor}" stroke="white" stroke-width="0.8" '
        f'data-uf="{sigla}" data-clientes="{clientes_uf}" class="estado" '
        f'onmouseover="showInfo(\'{sigla}\')" onclick="showInfo(\'{sigla}\')"/>\n'
    )

    cx, cy = centroid(feature['geometry'])
    if cx and cy:
        offset_y = OFFSET_Y.get(sigla, 0)
        offset_x = OFFSET_X.get(sigla, 0)
        labels_svg += (
            f'<text x="{cx + offset_x}" y="{cy + offset_y}" text-anchor="middle" '
            f'dominant-baseline="central" font-size="12" font-weight="700" '
            f'fill="#222" pointer-events="none" '
            f'style="text-shadow:0 0 4px white,0 0 4px white,0 0 4px white">{sigla}</text>\n'
        )

print(f"SVG gerado! Estados: {paths_svg.count('<path')}")
print(f"\nTop 5 estados por clientes:")
print(geo_dados.nlargest(5,'clientes')[['uf','clientes','renda_media']].to_string(index=False))

GeoJSON: 27 estados
Estados na base: 27
Colunas geo_dados: ['uf', 'clientes', 'renda_media', 'rfm_medio', 'champions', 'em_risco', 'heat', 'Old Money', 'Potencial Oculto', 'Primeiros Passos', 'Self Made', 'Trajetória Crescente']


SVG gerado! Estados: 27

Top 5 estados por clientes:
uf  clientes  renda_media
SP     64132        12097
SC     33338        11974
RJ     31637        12078
BA     25576        12183
MG     17633        12282


In [4]:
def cab(ws, row, colunas, cor_hex, height=32):
    fill = PatternFill(start_color=cor_hex, end_color=cor_hex, fill_type='solid')
    font = Font(bold=True, color='FFFFFF', size=10)
    for col_idx, col_nome in enumerate(colunas, 1):
        cell = ws.cell(row=row, column=col_idx, value=col_nome)
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    ws.row_dimensions[row].height = height

def tit(ws, text, n_cols, cor_hex='1a1a2e', height=30):
    ws.merge_cells(f'A1:{get_column_letter(n_cols)}1')
    ws['A1'].value = text
    ws['A1'].font = Font(bold=True, size=13, color='FFFFFF')
    ws['A1'].fill = PatternFill(start_color=cor_hex, end_color=cor_hex, fill_type='solid')
    ws['A1'].alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = height

def zebra(ws, row, valores, par=False):
    fill = PatternFill(start_color='F5F5F5', end_color='F5F5F5', fill_type='solid') if par else None
    for col_idx, valor in enumerate(valores, 1):
        cell = ws.cell(row=row, column=col_idx, value=valor)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        if fill:
            cell.fill = fill

def aj(ws, larguras):
    for col_idx, w in enumerate(larguras, 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = w

print("Funções auxiliares definidas!")

Funções auxiliares definidas!


In [5]:
COLUNAS_CLUSTER = [
    'ID Cliente','Nome','Profissão','Cluster','Idade','Renda (R$)','Score',
    'Produtos','Saldo Médio','Meses Rel.','Inadimplente',
    'Segmento RFM','Score RFM','Prioridade',
    'Produto Recomendado','Upgrade Recomendado',
    'Propensão Investimento','Propensão Previdência',
    'Propensão Cartão','Propensão Câmbio',
]
LARGURAS_CLUSTER = [14,24,24,22,8,14,8,10,14,10,12,22,10,14,26,26,20,20,16,16]

print("Gerando Excel...")
wb = openpyxl.Workbook()
wb.remove(wb.active)

# Aba 1 — Visão Geral
ws1 = wb.create_sheet('Visão Geral')
tit(ws1,'Motor de Recomendação — Visão Geral Executiva',4)
ws1.merge_cells('A2:D2')
ws1['A2'].value = f"Gerado em {datetime.now().strftime('%d/%m/%Y às %H:%M')} | Base: {total_clientes:,} clientes | K-Means 5 clusters"
ws1['A2'].font = Font(italic=True, size=10, color='555555')
ws1['A2'].alignment = Alignment(horizontal='center')
ws1.row_dimensions[2].height = 18
cab(ws1,3,['Métrica','Valor','Descrição',''],'EC0000')
kpis = [
    ('Total de Clientes',      f"{total_clientes:,}",      'Base consolidada'),
    ('Total Recomendações',    f"{total_recomendacoes:,}", 'Novos produtos + upgrades'),
    ('Clientes com Upgrade',   f"{clientes_upgrade:,}",    f"{clientes_upgrade/total_clientes:.1%} elegíveis"),
    ('Champions RFM',          f"{champions:,}",           'Score RFM ≥ 4.5'),
    ('Clientes em Risco',      f"{em_risco:,}",            'Score RFM ≤ 1.5'),
    ('Alta Prioridade',        f"{alta_prioridade:,}",     'Old Money + Champions'),
    ('Média Produtos/Cliente', f"{media_produtos:.1f}",    'Produtos base contratados'),
]
for i,(m,v,d) in enumerate(kpis):
    ws1.cell(i+4,1,value=m).font = Font(bold=True)
    ws1.cell(i+4,2,value=v).alignment = Alignment(horizontal='center')
    ws1.cell(i+4,3,value=d).font = Font(italic=True,color='555555')
    if i%2==1:
        for col in range(1,5):
            ws1.cell(i+4,col).fill = PatternFill(start_color='F5F5F5',end_color='F5F5F5',fill_type='solid')
    ws1.row_dimensions[i+4].height = 22
aj(ws1,[28,18,45,10])
ws1.freeze_panes = 'A3'

# Abas 2-6 — Clusters
for cluster_id in range(5):
    nome_aba = NOMES_CLUSTERS[cluster_id]
    cor_hex  = CORES_EXCEL[cluster_id]
    ws       = wb.create_sheet(nome_aba)
    df_c     = df[df['cluster_id']==cluster_id].copy()
    tit(ws, f"Cluster {cluster_id} — {nome_aba}", len(COLUNAS_CLUSTER), cor_hex)
    resumo = (f"Total: {len(df_c):,}  |  Renda média: R$ {df_c['renda_mensal'].mean():,.0f}  |  "
              f"Score médio: {df_c['score_credito'].mean():.0f}  |  "
              f"RFM médio: {df_c['rfm_score'].mean():.2f}  |  "
              f"Champions: {(df_c['rfm_segmento']=='Champions').sum():,}")
    n = len(COLUNAS_CLUSTER)
    ws.merge_cells(f'A2:{get_column_letter(n)}2')
    ws['A2'].value = resumo
    ws['A2'].font = Font(italic=True,size=10,color='555555')
    ws['A2'].alignment = Alignment(horizontal='center')
    ws.row_dimensions[2].height = 18
    cab(ws,3,COLUNAS_CLUSTER,cor_hex)
    for i,(_, row) in enumerate(df_c.iterrows()):
        valores = [
            row['id_cliente'], row.get('nome',''), row.get('profissao',''),
            nome_aba, int(row['idade']),
            round(row['renda_mensal'],2), int(row['score_credito']),
            int(row['qtd_produtos']), round(row['saldo_medio'],2),
            int(row['meses_relacionamento']),
            'Sim' if row['teve_inadimplencia_12m']==1 else 'Não',
            row.get('rfm_segmento',''),
            round(float(row.get('rfm_score',0)),2),
            row.get('prioridade_ataque',''),
            row.get('primeiro_produto_rec',''),
            row.get('primeiro_upgrade_rec',''),
            f"{row.get('propensao_investimento',0):.1f}%",
            f"{row.get('propensao_previdencia',0):.1f}%",
            f"{row.get('propensao_cartao_credito',0):.1f}%",
            f"{row.get('propensao_cambio',0):.1f}%",
        ]
        zebra(ws,i+4,valores,par=(i%2==1))
    aj(ws,LARGURAS_CLUSTER)
    ws.freeze_panes = 'A4'
    print(f"  Aba '{nome_aba}': {len(df_c):,} clientes")

# Aba 7 — Top Ataque
ws7 = wb.create_sheet('Top Ataque')
tit(ws7,'Top 100 Clientes para Ataque Comercial',9)
cab(ws7,2,['ID','Nome','Cluster','Renda (R$)','Score','RFM Score',
           'Segmento RFM','Prioridade','Produto Recomendado'],'EC0000')
top_ataque = df[df['qtd_novos_produtos']>0].nlargest(100,'renda_mensal')
for i,(_, row) in enumerate(top_ataque.iterrows()):
    cor=CORES_EXCEL[int(row['cluster_id'])]; nome=NOMES_CLUSTERS[int(row['cluster_id'])]; linha=i+3
    zebra(ws7,linha,[
        row['id_cliente'],row.get('nome',''),nome,
        round(row['renda_mensal'],2),int(row['score_credito']),
        round(float(row.get('rfm_score',0)),2),
        row.get('rfm_segmento',''),row.get('prioridade_ataque',''),
        row.get('primeiro_produto_rec',''),
    ],par=(i%2==1))
    ws7.cell(linha,3).fill=PatternFill(start_color=cor,end_color=cor,fill_type='solid')
    ws7.cell(linha,3).font=Font(color='FFFFFF',bold=True)
aj(ws7,[14,24,22,14,8,10,20,14,26]); ws7.freeze_panes='A3'

# Aba 8 — Upgrades
ws8 = wb.create_sheet('Upgrades')
tit(ws8,'Top 100 Clientes com Upgrades Disponíveis',8)
cab(ws8,2,['ID','Nome','Cluster','Renda (R$)','Score',
           'RFM Score','Upgrade Recomendado','Qtd Upgrades'],'9b59b6')
top_up = df[df['qtd_upgrades_rec']>0].nlargest(100,'renda_mensal')
for i,(_, row) in enumerate(top_up.iterrows()):
    cor=CORES_EXCEL[int(row['cluster_id'])]; nome=NOMES_CLUSTERS[int(row['cluster_id'])]; linha=i+3
    zebra(ws8,linha,[
        row['id_cliente'],row.get('nome',''),nome,
        round(row['renda_mensal'],2),int(row['score_credito']),
        round(float(row.get('rfm_score',0)),2),
        row.get('primeiro_upgrade_rec',''),
        int(row.get('qtd_upgrades_rec',0)),
    ],par=(i%2==1))
    ws8.cell(linha,3).fill=PatternFill(start_color=cor,end_color=cor,fill_type='solid')
    ws8.cell(linha,3).font=Font(color='FFFFFF',bold=True)
aj(ws8,[14,24,22,14,8,10,26,14]); ws8.freeze_panes='A3'

# Aba 9 — Propensão
ws9 = wb.create_sheet('Propensão por Cluster')
tit(ws9,'Score de Propensão Médio por Cluster (%)',len(cols_propensao)+1)
cols_prop = ['Cluster']+[c.replace('propensao_','').replace('_',' ').title() for c in cols_propensao]
cab(ws9,2,cols_prop,'f39c12')
for i,(cluster_nome,row) in enumerate(propensao_cluster.iterrows()):
    linha=i+3; cluster_id=[k for k,v in NOMES_CLUSTERS.items() if v==cluster_nome][0]
    cor=CORES_EXCEL[cluster_id]
    ws9.cell(linha,1,value=cluster_nome).fill=PatternFill(start_color=cor,end_color=cor,fill_type='solid')
    ws9.cell(linha,1).font=Font(color='FFFFFF',bold=True)
    ws9.cell(linha,1).alignment=Alignment(horizontal='center',vertical='center')
    for col_idx,val in enumerate(row.values,2):
        ws9.cell(linha,col_idx,value=f"{val:.1f}%").alignment=Alignment(horizontal='center',vertical='center')
    ws9.row_dimensions[linha].height=22
aj(ws9,[22]+[18]*len(cols_propensao))

# Aba 10 — Distribuição Geográfica
ws10 = wb.create_sheet('Distribuição Geográfica')
tit(ws10,'Distribuição Geográfica por Estado',6)
cab(ws10,2,['UF','Clientes','Renda Média (R$)','RFM Médio','Champions','Em Risco'],'3498db')
for i,row in enumerate(geo_dados.sort_values('clientes',ascending=False).itertuples()):
    linha=i+3
    zebra(ws10,linha,[row.uf,int(row.clientes),f"R$ {row.renda_media:,.0f}",
        f"{row.rfm_medio:.2f}",int(row.champions),int(row.em_risco)],par=(i%2==1))
aj(ws10,[8,12,18,12,12,12]); ws10.freeze_panes='A3'

wb.save(str(BASE_DIR/'docs'/'produto_final.xlsx'))
print(f"\nExcel salvo! Abas: {wb.sheetnames}")

Gerando Excel...


  Aba 'Primeiros Passos': 27,806 clientes


  Aba 'Trajetória Crescente': 98,729 clientes


  Aba 'Potencial Oculto': 94,225 clientes


  Aba 'Self Made': 64,079 clientes


  Aba 'Old Money': 15,161 clientes



Excel salvo! Abas: ['Visão Geral', 'Primeiros Passos', 'Trajetória Crescente', 'Potencial Oculto', 'Self Made', 'Old Money', 'Top Ataque', 'Upgrades', 'Propensão por Cluster', 'Distribuição Geográfica']


In [6]:
data_geracao = datetime.now().strftime('%d/%m/%Y às %H:%M')

# Gráfico de barras por cluster (padrão — antes de hover)
total_por_cluster = df.groupby(['cluster_id','cluster_nome']).size().reset_index(name='qtd')
total_por_cluster = total_por_cluster.sort_values('cluster_id')
max_qtd = total_por_cluster['qtd'].max()

CORES_CLUSTERS_HEX = {
    0: '#3498db',
    1: '#2ecc71',
    2: '#f39c12',
    3: '#9b59b6',
    4: '#e74c3c',
}

grafico_clusters_html = ''
for _, row in total_por_cluster.iterrows():
    cid  = int(row['cluster_id'])
    nome = row['cluster_nome']
    qtd  = int(row['qtd'])
    pct  = qtd / max_qtd * 100
    cor  = CORES_CLUSTERS_HEX[cid]
    grafico_clusters_html += f"""
    <div style="margin-bottom:10px">
      <div style="display:flex;justify-content:space-between;font-size:11px;margin-bottom:3px">
        <span style="font-weight:500;color:#333">{nome}</span>
        <span style="color:#888">{qtd:,}</span>
      </div>
      <div style="background:#f0f0f0;border-radius:4px;height:10px;overflow:hidden">
        <div style="width:{pct:.1f}%;height:100%;background:{cor};border-radius:4px"></div>
      </div>
    </div>"""

# Cards clusters
cards_clusters = ''
for _, row in metricas_cluster.iterrows():
    cluster_id = int(row['cluster_id'])
    cor = CORES_CLUSTERS[cluster_id]
    cards_clusters += f"""
    <div class="cluster-card">
        <div class="cluster-header" style="background:{cor}">
            <span class="cluster-badge">Cluster {cluster_id}</span>
            <h3>{row['cluster_nome']}</h3>
            <span class="cluster-count">{int(row['qtd_clientes']):,} clientes</span>
        </div>
        <div class="cluster-body">
            <div class="metric-row">
                <div class="metric"><span class="label">Renda média</span>
                    <span class="value">R$ {row['renda_media']:,.0f}</span></div>
                <div class="metric"><span class="label">Score médio</span>
                    <span class="value">{row['score_medio']:.0f}</span></div>
            </div>
            <div class="metric-row">
                <div class="metric"><span class="label">RFM médio</span>
                    <span class="value">{row['rfm_score_medio']:.2f}</span></div>
                <div class="metric"><span class="label">Champions</span>
                    <span class="value">{row['champions_pct']:.1%}</span></div>
            </div>
            <div class="metric-row">
                <div class="metric"><span class="label">Em risco</span>
                    <span class="value">{row['em_risco_pct']:.1%}</span></div>
                <div class="metric"><span class="label">Inadimplência</span>
                    <span class="value">{row['taxa_inadimplencia']:.1%}</span></div>
            </div>
        </div>
    </div>"""

# Tabela propensão
header_prop = ''.join([f"<th>{c.replace('propensao_','').replace('_',' ').title()}</th>" for c in cols_propensao])
linhas_prop = ''
for cluster_nome, row in propensao_cluster.iterrows():
    cluster_id = [k for k,v in NOMES_CLUSTERS.items() if v==cluster_nome][0]
    cor = CORES_CLUSTERS[cluster_id]
    cels = ''.join([f"<td>{v:.1f}%</td>" for v in row.values])
    linhas_prop += f"""
    <tr>
        <td><span class="tag" style="background:{cor}20;color:{cor};border:1px solid {cor}40">{cluster_nome}</span></td>
        {cels}
    </tr>"""

# Top ataque
linhas_ataque = ''
for cluster_id in range(5):
    cor  = CORES_CLUSTERS[cluster_id]
    nome = NOMES_CLUSTERS[cluster_id]
    top  = df[(df['cluster_id']==cluster_id)&(df['qtd_novos_produtos']>0)].nlargest(15,'renda_mensal')
    linhas = ''
    for _, r in top.iterrows():
        prio = r.get('prioridade_ataque','')
        prio_cor = '#27ae60' if prio=='Alta' else '#f39c12' if prio=='Média' else '#95a5a6'
        linhas += f"""
        <tr>
            <td><code>{r['id_cliente']}</code></td>
            <td>{r.get('nome','')}</td>
            <td>{r.get('profissao','')}</td>
            <td>R$ {r['renda_mensal']:,.0f}</td>
            <td>{int(r['score_credito'])}</td>
            <td>{r.get('rfm_score',0):.2f}</td>
            <td><span class="tag" style="background:{prio_cor}20;color:{prio_cor};border:1px solid {prio_cor}40">{prio}</span></td>
            <td><span class="tag" style="background:{cor}20;color:{cor};border:1px solid {cor}40">{r.get('primeiro_produto_rec','')}</span></td>
            <td><span class="badge">{int(r.get('qtd_novos_produtos',0))}</span></td>
        </tr>"""
    linhas_ataque += f"""
    <div class="section-cluster" style="border-top:3px solid {cor}">
        <h4 style="color:{cor}">Cluster {cluster_id} — {nome}</h4>
        <table><thead><tr>
            <th>ID</th><th>Nome</th><th>Profissão</th><th>Renda</th><th>Score</th>
            <th>RFM</th><th>Prioridade</th><th>1ª Recomendação</th><th>Total</th>
        </tr></thead><tbody>{linhas}</tbody></table>
    </div>"""

# Top upgrades
linhas_upgrades = ''
for cluster_id in range(5):
    cor  = CORES_CLUSTERS[cluster_id]
    nome = NOMES_CLUSTERS[cluster_id]
    top  = df[(df['cluster_id']==cluster_id)&(df['qtd_upgrades_rec']>0)].nlargest(10,'renda_mensal')
    if len(top)==0: continue
    linhas = ''
    for _, r in top.iterrows():
        linhas += f"""
        <tr>
            <td><code>{r['id_cliente']}</code></td>
            <td>{r.get('nome','')}</td>
            <td>R$ {r['renda_mensal']:,.0f}</td>
            <td>{int(r['score_credito'])}</td>
            <td>{r.get('rfm_score',0):.2f}</td>
            <td><span class="tag" style="background:{cor}20;color:{cor};border:1px solid {cor}40">{r.get('primeiro_upgrade_rec','')}</span></td>
            <td><span class="badge">{int(r.get('qtd_upgrades_rec',0))}</span></td>
        </tr>"""
    linhas_upgrades += f"""
    <div class="section-cluster" style="border-top:3px solid {cor}">
        <h4 style="color:{cor}">Cluster {cluster_id} — {nome}</h4>
        <table><thead><tr>
            <th>ID</th><th>Nome</th><th>Renda</th><th>Score</th>
            <th>RFM</th><th>Upgrade Recomendado</th><th>Total</th>
        </tr></thead><tbody>{linhas}</tbody></table>
    </div>"""

# Ranking
itens_ranking = ''.join([
    f"""<div class="rank-item">
        <div class="rank-bar-wrap">
            <div class="rank-label">{row['produto']}</div>
            <div class="rank-bar" style="width:{min(row['pct']*2.5,100):.0f}%;background:#e74c3c"></div>
        </div>
        <div class="rank-value">{int(row['quantidade']):,}<span> ({row['pct']:.1f}%)</span></div>
    </div>"""
    for i,row in ranking_produtos.iterrows()
])

itens_upgrades_rank = ''.join([
    f"""<div class="rank-item">
        <div class="rank-bar-wrap">
            <div class="rank-label">{row['upgrade']}</div>
            <div class="rank-bar" style="width:{min(row['pct']*2,100):.0f}%;background:#9b59b6"></div>
        </div>
        <div class="rank-value">{int(row['quantidade']):,}<span> ({row['pct']:.1f}%)</span></div>
    </div>"""
    for i,row in ranking_upgrades.iterrows()
])

html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Motor de Recomendação de Produtos</title>
<style>
  *{{margin:0;padding:0;box-sizing:border-box}}
  body{{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;background:#f0f2f5;color:#1a1a2e;font-size:14px}}
  .header{{background:linear-gradient(135deg,#EC0000 0%,#a30000 100%);padding:24px 40px;color:white;position:sticky;top:0;z-index:100;box-shadow:0 2px 8px rgba(0,0,0,0.3)}}
  .header-top{{display:flex;justify-content:space-between;align-items:center}}
  .logo{{font-size:18px;font-weight:800;display:flex;align-items:center;gap:8px}}
  .logo-dot{{width:9px;height:9px;background:white;border-radius:50%;flex-shrink:0}}
  .header h1{{margin-top:6px;font-size:19px;font-weight:600}}
  .header p{{margin-top:3px;opacity:0.75;font-size:12px}}
  .data-badge{{background:rgba(255,255,255,0.2);padding:4px 12px;border-radius:20px;font-size:11px;white-space:nowrap}}
  .nav{{background:white;border-bottom:2px solid #f0f0f0;padding:0 40px;display:flex;position:sticky;top:78px;z-index:99;overflow-x:auto}}
  .nav a{{padding:12px 16px;font-size:13px;font-weight:500;color:#888;text-decoration:none;border-bottom:3px solid transparent;transition:all 0.2s;white-space:nowrap;margin-bottom:-2px}}
  .nav a:hover,.nav a.active{{color:#e74c3c;border-bottom-color:#e74c3c}}
  .container{{max-width:1200px;margin:0 auto;padding:20px}}
  .section{{background:white;border-radius:12px;padding:22px 26px;margin-bottom:20px;box-shadow:0 2px 8px rgba(0,0,0,0.06)}}
  .section h2{{font-size:16px;font-weight:600;margin-bottom:16px;padding-bottom:10px;border-bottom:2px solid #f0f0f0;display:flex;align-items:center;gap:8px}}
  .section h2::before{{content:'';display:block;width:4px;height:16px;background:#e74c3c;border-radius:2px;flex-shrink:0}}
  .section h4{{font-size:14px;font-weight:600;margin:14px 0 8px}}
  .kpi-grid{{display:grid;grid-template-columns:repeat(4,1fr);gap:12px}}
  .kpi{{background:#fafafa;border-radius:8px;padding:16px;border-left:4px solid #e74c3c}}
  .kpi-value{{font-size:22px;font-weight:700;color:#e74c3c}}
  .kpi-label{{font-size:12px;color:#666;margin-top:3px}}
  .kpi-sub{{font-size:11px;color:#aaa;margin-top:1px}}
  .clusters-grid{{display:grid;grid-template-columns:repeat(3,1fr);gap:12px}}
  .cluster-card{{border-radius:10px;overflow:hidden;box-shadow:0 2px 6px rgba(0,0,0,0.07)}}
  .cluster-header{{padding:13px 16px;color:white}}
  .cluster-header h3{{font-size:14px;font-weight:600;margin:3px 0}}
  .cluster-badge{{font-size:10px;opacity:0.85;text-transform:uppercase;letter-spacing:0.5px}}
  .cluster-count{{font-size:12px;opacity:0.9}}
  .cluster-body{{padding:13px 16px;background:white}}
  .metric-row{{display:flex;gap:10px;margin-bottom:6px}}
  .metric{{flex:1}}
  .metric .label{{font-size:10px;color:#888;display:block}}
  .metric .value{{font-size:13px;font-weight:600}}
  .mapa-wrap{{display:grid;grid-template-columns:1fr 280px;gap:16px;align-items:center}}
  .mapa-box{{background:#f8f8f8;border-radius:10px;padding:10px}}
  .estado{{transition:fill 0.3s;cursor:pointer}}
  .estado:hover{{opacity:0.75;stroke-width:1.5!important}}
  .legenda-bar{{height:9px;border-radius:4px;background:linear-gradient(to right,#e0e0e0,#e74c3c);margin:6px 0}}
  .legenda-labels{{display:flex;justify-content:space-between;font-size:10px;color:#aaa}}
  .info-box{{background:#f8f8f8;border-radius:8px;padding:14px;min-height:80px}}
  .info-box h4{{font-size:13px;font-weight:600;margin-bottom:8px}}
  .info-row{{display:flex;justify-content:space-between;padding:4px 0;border-bottom:1px solid #eee;font-size:12px}}
  .info-row:last-child{{border-bottom:none}}
  .info-label{{color:#888}}
  .info-value{{font-weight:600}}
  .section-cluster{{margin-bottom:18px;padding-bottom:18px;border-bottom:1px solid #f0f0f0}}
  .section-cluster:last-child{{border-bottom:none;margin-bottom:0;padding-bottom:0}}
  table{{width:100%;border-collapse:collapse;font-size:12px}}
  thead tr{{background:#f8f8f8}}
  th{{padding:8px 10px;text-align:left;font-weight:600;color:#555;font-size:11px;text-transform:uppercase;letter-spacing:0.3px}}
  td{{padding:7px 10px;border-bottom:1px solid #f5f5f5}}
  tr:hover td{{background:#fafafa}}
  code{{background:#f0f0f0;padding:2px 5px;border-radius:3px;font-size:11px;color:#555}}
  .tag{{padding:2px 8px;border-radius:20px;font-size:11px;font-weight:500}}
  .badge{{background:#e74c3c;color:white;padding:1px 7px;border-radius:10px;font-size:11px;font-weight:600}}
  .ranking-grid{{display:grid;grid-template-columns:1fr 1fr;gap:24px}}
  .rank-item{{display:flex;align-items:center;gap:8px;margin-bottom:9px}}
  .rank-bar-wrap{{flex:1}}
  .rank-label{{font-size:12px;margin-bottom:3px;color:#333}}
  .rank-bar{{height:7px;border-radius:3px;min-width:3px}}
  .rank-value{{font-size:12px;font-weight:600;min-width:80px;text-align:right;color:#333}}
  .rank-value span{{font-weight:400;color:#aaa;font-size:11px}}
  .prop-table th,.prop-table td{{text-align:center}}
  .footer{{text-align:center;padding:16px;color:#ccc;font-size:11px}}
</style>
</head>
<body>

<div class="header">
  <div class="header-top">
    <div class="logo"><div class="logo-dot"></div>Analytics <span style="font-weight:300;opacity:0.75;margin-left:6px;font-size:15px">Motor de Recomendação</span></div>
    <div class="data-badge">Gerado em {data_geracao}</div>
  </div>
  <h1>Motor de Recomendação de Produtos</h1>
  <p>K-Means · RFM Score · Score de Propensão · De-Para · Pipeline Incremental · {total_clientes:,} clientes · 5 clusters</p>
</div>

<nav class="nav">
  <a href="#mapa" class="active">Mapa</a>
  <a href="#visao-geral">Visão Geral</a>
  <a href="#segmentacao">Segmentação</a>
  <a href="#propensao">Propensão</a>
  <a href="#ranking">Ranking</a>
  <a href="#top-ataque">Top Ataque</a>
  <a href="#upgrades">Upgrades</a>
</nav>

<div class="container">

  <div class="section" id="mapa">
    <h2>Distribuição Geográfica — Mapa de Calor por Clientes</h2>
    <div class="mapa-wrap" style="align-items:stretch">
      <div style="display:flex;flex-direction:column;justify-content:space-between">
        <div class="mapa-box" style="flex:1">
          <svg id="brasil-map" viewBox="0 0 520 540" width="100%" style="display:block;max-height:440px">
            {paths_svg}
            {labels_svg}
          </svg>
        </div>
        <div style="margin-top:10px">
          <div style="font-size:10px;color:#aaa;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px">Escala — Concentração de clientes</div>
          <div class="legenda-bar"></div>
          <div class="legenda-labels"><span>Menos clientes</span><span>Mais clientes</span></div>
        </div>
      </div>
<div style="display:flex;flex-direction:column">
        <div class="info-box" id="mapa-info" style="flex:1;display:flex;flex-direction:column;justify-content:space-between">
          <div>
            <h4 style="color:#ccc;font-weight:400;font-size:12px">Passe o mouse sobre um estado</h4>
            <div style="margin-top:12px">
              <div style="font-size:10px;font-weight:600;color:#aaa;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:8px">Clientes por Perfil — Brasil</div>
              {grafico_clusters_html}
            </div>
          </div>
        </div>
      </div>
    </div>
  </div>

  <div class="section" id="visao-geral">
    <h2>Visão Geral Executiva</h2>
    <div class="kpi-grid">
      <div class="kpi"><div class="kpi-value">{total_clientes:,}</div><div class="kpi-label">Clientes analisados</div><div class="kpi-sub">Base consolidada</div></div>
      <div class="kpi"><div class="kpi-value">{total_recomendacoes:,}</div><div class="kpi-label">Recomendações geradas</div><div class="kpi-sub">Novos produtos + upgrades</div></div>
      <div class="kpi"><div class="kpi-value">{champions:,}</div><div class="kpi-label">Champions RFM</div><div class="kpi-sub">Score RFM ≥ 4.5</div></div>
      <div class="kpi"><div class="kpi-value">{alta_prioridade:,}</div><div class="kpi-label">Alta prioridade ataque</div><div class="kpi-sub">Old Money + Champions</div></div>
      <div class="kpi"><div class="kpi-value">{clientes_upgrade:,}</div><div class="kpi-label">Elegíveis para upgrade</div><div class="kpi-sub">{clientes_upgrade/total_clientes:.1%} da base</div></div>
      <div class="kpi"><div class="kpi-value">{em_risco:,}</div><div class="kpi-label">Clientes em risco</div><div class="kpi-sub">Retenção urgente</div></div>
      <div class="kpi"><div class="kpi-value">{media_produtos:.1f}</div><div class="kpi-label">Média produtos/cliente</div><div class="kpi-sub">Produtos base contratados</div></div>
      <div class="kpi"><div class="kpi-value">5</div><div class="kpi-label">Clusters identificados</div><div class="kpi-sub">K-Means validado</div></div>
    </div>
  </div>

  <div class="section" id="segmentacao">
    <h2>Segmentação de Clientes</h2>
    <div class="clusters-grid">{cards_clusters}</div>
  </div>

  <div class="section" id="propensao">
    <h2>Score de Propensão por Cluster (%)</h2>
    <p style="color:#888;font-size:12px;margin-bottom:12px">Regressão Logística — AUC-ROC médio acima de 0.80 em todos os produtos.</p>
    <table class="prop-table">
      <thead><tr><th>Cluster</th>{header_prop}</tr></thead>
      <tbody>{linhas_prop}</tbody>
    </table>
  </div>

  <div class="section" id="ranking">
    <h2>Ranking de Recomendações</h2>
    <div class="ranking-grid">
      <div><h4>Novos produtos mais recomendados</h4>{itens_ranking}</div>
      <div><h4>Upgrades mais recomendados</h4>{itens_upgrades_rank}</div>
    </div>
  </div>

  <div class="section" id="top-ataque">
    <h2>Top Clientes para Ataque Comercial</h2>
    <p style="color:#888;margin-bottom:14px;font-size:12px">15 clientes de maior renda por cluster com recomendações ativas.</p>
    {linhas_ataque}
  </div>

  <div class="section" id="upgrades">
    <h2>Clientes com Upgrades Disponíveis</h2>
    <p style="color:#888;margin-bottom:14px;font-size:12px">10 clientes de maior renda por cluster elegíveis para upgrade.</p>
    {linhas_upgrades}
  </div>

</div>

<div class="footer">Motor de Recomendação · {data_geracao} · Dados sintéticos para fins de demonstração</div>

<script>
const geoData = {geo_json_str};
const nomeEstados = {{
  AC:'Acre',AL:'Alagoas',AP:'Amapá',AM:'Amazonas',BA:'Bahia',
  CE:'Ceará',DF:'Distrito Federal',ES:'Espírito Santo',GO:'Goiás',
  MA:'Maranhão',MT:'Mato Grosso',MS:'Mato Grosso do Sul',MG:'Minas Gerais',
  PA:'Pará',PB:'Paraíba',PR:'Paraná',PE:'Pernambuco',PI:'Piauí',
  RJ:'Rio de Janeiro',RN:'Rio Grande do Norte',RS:'Rio Grande do Sul',
  RO:'Rondônia',RR:'Roraima',SC:'Santa Catarina',SP:'São Paulo',
  SE:'Sergipe',TO:'Tocantins'
}};

function showInfo(uf) {{
  const d = geoData.find(x => x.uf === uf);
  if (!d) return;

  const clusters = [
    {{nome:'Primeiros Passos',    cor:'#3498db', key:'Primeiros Passos'}},
    {{nome:'Trajetória Crescente',cor:'#2ecc71', key:'Trajetória Crescente'}},
    {{nome:'Potencial Oculto',    cor:'#f39c12', key:'Potencial Oculto'}},
    {{nome:'Self Made',           cor:'#9b59b6', key:'Self Made'}},
    {{nome:'Old Money',           cor:'#e74c3c', key:'Old Money'}},
  ];

  const maxQtd = Math.max(...clusters.map(c => d[c.key] || 0));

  const barras = clusters.map(c => {{
    const qtd = d[c.key] || 0;
    const pct = maxQtd > 0 ? (qtd / maxQtd * 100).toFixed(1) : 0;
    return `
      <div style="margin-bottom:7px">
        <div style="display:flex;justify-content:space-between;font-size:10px;margin-bottom:2px">
          <span style="font-weight:500;color:#333">${{c.nome}}</span>
          <span style="color:#888">${{qtd.toLocaleString('pt-BR')}}</span>
        </div>
        <div style="background:#f0f0f0;border-radius:3px;height:8px;overflow:hidden">
          <div style="width:${{pct}}%;height:100%;background:${{c.cor}};border-radius:3px"></div>
        </div>
      </div>`;
  }}).join('');

  document.getElementById('mapa-info').innerHTML = `
    <h4 style="margin-bottom:6px">${{nomeEstados[uf]||uf}} (${{uf}})</h4>
    <div class="info-row"><span class="info-label">Total Clientes</span>
      <span class="info-value">${{d.clientes.toLocaleString('pt-BR')}}</span></div>
    <div class="info-row"><span class="info-label">Renda Média</span>
      <span class="info-value">R$ ${{Math.round(d.renda_media).toLocaleString('pt-BR')}}</span></div>
    <div class="info-row" style="margin-bottom:8px"><span class="info-label">RFM Médio</span>
      <span class="info-value">${{d.rfm_medio.toFixed(2)}}</span></div>
<div style="font-size:10px;font-weight:600;color:#555;text-transform:uppercase;letter-spacing:0.5px;margin:20px 0 8px;padding-top:12px;border-top:1px solid #eee">Distribuição por Perfil</div>
    ${{barras}}`;
}}

const navLinks = document.querySelectorAll('.nav a');
window.addEventListener('scroll', () => {{
  let current = 'mapa';
  document.querySelectorAll('.section').forEach(s => {{
    if (window.scrollY >= s.offsetTop - 140) current = s.id;
  }});
  navLinks.forEach(a => {{
    a.classList.remove('active');
    if (a.getAttribute('href') === '#' + current) a.classList.add('active');
  }});
}});
</script>
</body>
</html>"""

with open(str(BASE_DIR/'docs'/'relatorio_final.html'), 'w', encoding='utf-8') as f:
    f.write(html)

print("HTML salvo!")

HTML salvo!


In [7]:
def ler_env():
    env_path = BASE_DIR / '.env'
    variaveis = {}
    with open(env_path, 'r', encoding='utf-8') as f:
        for linha in f:
            linha = linha.strip()
            if '=' in linha and not linha.startswith('#'):
                chave, valor = linha.split('=', 1)
                variaveis[chave.strip()] = valor.strip()
    return variaveis

env        = ler_env()
EMAIL_REM  = env.get('EMAIL_REMETENTE','')
EMAIL_DEST = env.get('EMAIL_DESTINATARIO','')
MTRAP_HOST = env.get('MAILTRAP_HOST','')
MTRAP_PORT = int(env.get('MAILTRAP_PORT',587))
MTRAP_USER = env.get('MAILTRAP_USER','')
MTRAP_PASS = env.get('MAILTRAP_PASS','')

def enviar():
    print("Preparando envio...")
    msg = MIMEMultipart('mixed')
    msg['Subject'] = f'[Analytics] Relatório Final Motor de Recomendação — {datetime.now().strftime("%d/%m/%Y")}'
    msg['From']    = EMAIL_REM
    msg['To']      = EMAIL_DEST

    with open(str(BASE_DIR/'docs'/'relatorio_final.html'), 'r', encoding='utf-8') as f:
        msg.attach(MIMEText(f.read(), 'html'))
    print("HTML anexado!")

    # Excel resumido para email — top 20 por cluster
    wb_email = openpyxl.Workbook()
    wb_email.remove(wb_email.active)
    for cluster_id in range(5):
        nome_aba = NOMES_CLUSTERS[cluster_id]
        cor_hex  = CORES_EXCEL[cluster_id]
        ws_e     = wb_email.create_sheet(title=nome_aba)
        df_c     = df[df['cluster_id']==cluster_id].nlargest(20,'renda_mensal')
        cols_r   = ['ID','Nome','Profissão','Renda (R$)','Score','Segmento RFM','Prioridade','Produto Recomendado','Upgrade']
        tit(ws_e, f"{nome_aba} — Top 20", len(cols_r), cor_hex)
        cab(ws_e, 2, cols_r, cor_hex)
        for i,(_, row) in enumerate(df_c.iterrows()):
            zebra(ws_e,i+3,[
                row['id_cliente'],row.get('nome',''),row.get('profissao',''),
                round(row['renda_mensal'],2),int(row['score_credito']),
                row.get('rfm_segmento',''),row.get('prioridade_ataque',''),
                row.get('primeiro_produto_rec',''),row.get('primeiro_upgrade_rec',''),
            ],par=(i%2==1))
        aj(ws_e,[14,24,24,14,8,20,14,26,26])

    buf = io.BytesIO()
    wb_email.save(buf)
    buf.seek(0)
    print(f"Excel resumido: {buf.getbuffer().nbytes/1024/1024:.2f} MB")

    parte = MIMEBase('application','octet-stream')
    parte.set_payload(buf.read())
    encoders.encode_base64(parte)
    parte.add_header('Content-Disposition','attachment',filename='top_clientes.xlsx')
    msg.attach(parte)
    print("Excel anexado!")

    with smtplib.SMTP(MTRAP_HOST, MTRAP_PORT) as server:
        server.ehlo()
        server.starttls()
        server.login(MTRAP_USER, MTRAP_PASS)
        server.sendmail(EMAIL_REM, EMAIL_DEST, msg.as_string())
        print(f"Email enviado para {EMAIL_DEST}!")

enviar()

Preparando envio...
HTML anexado!


Excel resumido: 0.01 MB
Excel anexado!


Email enviado para calebe.bizarro@outlook.com!
